## IMPORTS


In [8]:
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os
import pandas as pd


In [2]:
PREDICT = 144*7
data = pd.read_csv("../powerconsumption.csv")
data["Datetime"] = pd.to_datetime(data["Datetime"])
data = data.sort_values("Datetime").set_index("Datetime")
data['target'] = data["PowerConsumption_Zone1"]
ready = data["target"].dropna()

TARGET = ready

In [3]:
class Target(Dataset):
    def __init__(self, data, seq_len, pred_len):
        super().__init__()
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.tensor(x).unsqueeze(-1), torch.tensor(y).unsqueeze(-1)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)             # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return x

In [ ]:
class TransformerForecaster(nn.Module):
    def __init__(
        self,
        pred_len,
        seq_len,
        input_size=1,
        d_model=32,          
        nhead=4,
        num_encoder_layers=2, 
        num_decoder_layers=2, 
        dim_feedforward=128, 
    ):
        super().__init__()
        self.pred_len = pred_len
        self.d_model = d_model

        # Proyectan la serie al espacio interno del Transformer
        # Si input_size=1 y d_model=64, convierte cada valor escalar en un vector de 64 dimensiones
        self.src_projection = nn.Linear(input_size, d_model)  # Para la entrada histórica (encoder)
        self.tgt_projection = nn.Linear(input_size, d_model)  # Para el objetivo (decoder)

        # Añade información de posición temporal (ver clase PositionalEncoding)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max(seq_len, pred_len))

        self.transformer = nn.Transformer(
            d_model=d_model,                        # Debe coincidir con el d_model de arriba
            nhead=nhead,                            # Debe dividir a d_model
            num_encoder_layers=num_encoder_layers,  # Profundidad del encoder
            num_decoder_layers=num_decoder_layers,  # Profundidad del decoder
            dim_feedforward=dim_feedforward,        # Ancho de la red interna
            batch_first=True,                       # Mantén esto en True — hace que la forma sea (Batch, Seq, Features)
        )

        # Proyecta de vuelta al espacio original para obtener la predicción final
        # d_model=64 → input_size=1, o sea, de vector de 64 dims a un valor escalar
        self.output_projection = nn.Linear(d_model, input_size)

    def forward(self, src, tgt):
        # src: (B, seq_len, input_size) — ventana histórica que ve el encoder
        # tgt: (B, pred_len, input_size) — pasos objetivo que ve el decoder (con teacher forcing)

        # Proyectar + añadir positional encoding a encoder y decoder
        src = self.pos_encoder(self.src_projection(src))
        tgt = self.pos_encoder(self.tgt_projection(tgt))

        # Máscara causal: impide que el decoder vea pasos futuros durante entrenamiento
        # Es una matriz triangular superior de -inf — no la toques, es estándar
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1), device=src.device
        )

        out = self.transformer(src, tgt, tgt_mask=tgt_mask)

        # Salida final: (B, pred_len, input_size) — la predicción para cada paso futuro
        return self.output_projection(out)


In [5]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Teacher forcing: el decoder recibe el último valor del encoder
        # + todos los pasos objetivo menos el último
        tgt_input = torch.cat([x[:, -1:, :], y[:, :-1, :]], dim=1)

        optimizer.zero_grad()
        pred = model(x, tgt_input)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [6]:
@torch.no_grad()
def predict(model, src, pred_len, device):
    model.eval()
    src = src.to(device)
    # Arrancamos el decoder con el último valor conocido
    tgt = src[:, -1:, :]

    preds = []
    for _ in range(pred_len):
        out = model(src, tgt)
        next_val = out[:, -1:, :]
        preds.append(next_val)
        tgt = torch.cat([tgt, next_val], dim=1)

    return torch.cat(preds, dim=1).squeeze(-1).cpu().numpy()


In [ ]:
if __name__ == "__main__":
    DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
    SEQ_LEN  = 144*7*4  # 1 mes de historia
    """
    MENCIONAR EN EL DOCUMENTO QUE HABIAMOS PROBADO A ENTRENAR AL MODELO CON MAS MESES PERO PETA POR MEMORIA INSUFICIENTE EN LA RAM
    """
    PRED_LEN = PREDICT        
    EPOCHS   = 30
    BATCH    = 8
    MODEL_PATH = "transformer_forecaster.pth"  # ✅ ruta donde se guarda/carga

    series = TARGET.values.astype("float32")  
    split  = int(len(series) * 0.8)
    train_ds = Target(series[:split], SEQ_LEN, PRED_LEN)
    test_ds  = Target(series[split:],  SEQ_LEN, PRED_LEN)

    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

    # Modelo
    model = TransformerForecaster(pred_len=PRED_LEN, seq_len=SEQ_LEN).to(DEVICE)

    # ✅ Si ya existe un modelo guardado, lo carga y se salta el entrenamiento
    if os.path.exists(MODEL_PATH):
        print(f"Modelo encontrado en '{MODEL_PATH}', cargando pesos...")
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print("Modelo cargado. Saltando entrenamiento.")
    else:
        print("No se encontró modelo guardado. Entrenando desde cero...")
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
        criterion = nn.MSELoss()

        for epoch in range(1, EPOCHS + 1):
            loss = train(model, train_dl, optimizer, criterion, DEVICE)
            scheduler.step()
            if epoch % 5 == 0:
                print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss:.6f}")

        # ✅ Guarda el modelo tras entrenar
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Modelo guardado en '{MODEL_PATH}'.")

    # Evaluación visual
    x_test, y_test = test_ds[0]
    src = x_test.unsqueeze(0)
    preds = predict(model, src, PRED_LEN, DEVICE)

    plt.figure(figsize=(12, 4))
    t_hist = np.arange(SEQ_LEN)
    t_pred = np.arange(SEQ_LEN, SEQ_LEN + PRED_LEN)
    plt.plot(t_hist, x_test.squeeze().numpy(), label="Historia")
    plt.plot(t_pred, y_test.squeeze().numpy(), label="Real", linestyle="--")
    plt.plot(t_pred, preds[0],                 label="Predicción", linestyle=":")
    plt.legend()
    plt.title("Transformer Forecasting")
    plt.tight_layout()
    plt.savefig("forecast.png", dpi=150)
    plt.show()

No se encontró modelo guardado. Entrenando desde cero...
